In [33]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
import optuna
from optuna.pruners import MedianPruner
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from google.colab import drive
drive.mount('/content/drive')


letter_recognition = fetch_ucirepo(id=59)
letter_recognition = pd.concat([letter_recognition.data.features, letter_recognition.data.targets], axis=1)
letter_recognition.lettr.value_counts()
dummies = pd.get_dummies(letter_recognition['lettr'], drop_first=True, prefix='lettr').astype(int)
letter_recognition = pd.concat([letter_recognition, dummies], axis=1)
letter_recognition.drop(columns=['lettr'], inplace=True)

letter_recognition = letter_recognition.dropna()
letter_recognition

X = letter_recognition.drop(columns=['high'])  # все столбцы, кроме 'high'
y = letter_recognition['high']                 # целевая переменная

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Val:   {X_val.shape}, {y_val.shape}")
print(f"Test:  {X_test.shape}, {y_test.shape}")

model = CatBoostRegressor(
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=4,
    random_strength=4,
    bagging_temperature=0.600000001,
    grow_policy='Depthwise',
    n_estimators=500,
    loss_function='RMSE',
    verbose=False
    )

# Обучение на train
model.fit(pd.concat([X_train, X_val], axis=0), pd.concat([y_train, y_val], axis=0))

# Предсказание на val
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R² на тесте: {r2:.3f}")
print(f"MAE на тесте: {mae:.3f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (12000, 40), (12000,)
Val:   (4000, 40), (4000,)
Test:  (4000, 40), (4000,)
R² на тесте: 0.951
MAPE на тесте: 0.365


In [ ]:
def objective(trial):

    # Гиперпараметры CatBoost
    params = {
        'learning_rate': trial.suggest_categorical('learning_rate', [0.1, 0.01]),
        'depth': trial.suggest_categorical('depth', [2, 4, 6]),                                # Глубина деревьев
        'l2_leaf_reg': trial.suggest_categorical('l2_leaf_reg', [0, 2, 4, 6, 10]),         # Регуляризация L2
        'random_strength': trial.suggest_float('random_strength', 0, 10.0, step=2),  # Случайность при построении деревьев
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1.0, step=0.2),      # Температура бэггинга
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),  # Политика роста деревьев
        'n_estimators': trial.suggest_categorical('n_estimators', [100, 200, 500]),       # Количество деревьев
        'loss_function': trial.suggest_categorical('loss_function', ['RMSE', 'MAE', 'Quantile', 'MAPE']),  # Функция потерь
        'verbose': False,          # Отключаем вывод в консоль
        'random_state': 42         # Фиксируем случайность
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return r2_score(y_val, y_pred)  # Optuna максимизирует R²

study = optuna.create_study(
    direction='maximize', # Максимизируем R²
    load_if_exists=True,
    storage="sqlite:////content/drive/MyDrive/database2/params_catboost.db",
    study_name="params_catboost"
)

total_trials = 1000
chunk_size = 500

study.optimize(
    objective,
    n_trials=1000,
    show_progress_bar=True,
    gc_after_trial=True,
    n_jobs=-1
)

In [26]:
trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values('value', ascending=False)
trials_df.to_csv('/content/drive/MyDrive/database.csv')
trials_df
# Лучшие параметры
print("Лучшие параметры:", study.best_params)
print("Лучший R²:", study.best_value)

Лучшие параметры: {'learning_rate': 0.1, 'depth': 6, 'l2_leaf_reg': 0, 'random_strength': 4.0, 'bagging_temperature': 0.6000000000000001, 'grow_policy': 'Depthwise', 'n_estimators': 500, 'loss_function': 'RMSE'}
Лучший R²: 0.9519076901544803
